In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 11 consolidation — Build a feature pipeline without seeing the answer

Read `sessions/11_session.md` first. This is an executed reference, not an assessment of you.

**Evidence:** Real train/validation digit pixels and the bundled historical annual sunspot series; no new forecast claim.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'data/raw/iris.csv').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Start Jupyter in the extracted listening_to_shape_lab folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))


## Step 1

Fit a transformation only on the training digit pixels.

Before running: write your expected result and name one way this check could fail.

In [3]:
from shape_lab.data import load_digit_data
from sklearn.preprocessing import StandardScaler
train, _, _ = load_digit_data('train')
valid, _, _ = load_digit_data('validation')
X_train = train.reshape(len(train), -1)
X_valid = valid.reshape(len(valid), -1)
scaler = StandardScaler().fit(X_train)
assert np.allclose(scaler.mean_, X_train.mean(axis=0))
Z_valid = scaler.transform(X_valid)
print('training rows:', len(train), 'validation rows:', len(valid))
print('mean absolute validation-feature mean:', np.abs(Z_valid.mean(0)).mean())

training rows: 1077 validation rows: 360
mean absolute validation-feature mean: 0.03951193000527718


## Step 2

Build windows from a real historical time series and attach the last included year.

Before running: write your expected result and name one way this check could fail.

In [4]:
import pandas as pd
from shape_lab.features import delay_embedding
series = pd.read_csv(ROOT/'data/time_series/sunspots_yearly.csv')
values = series.SUNACTIVITY.to_numpy()
windows = delay_embedding(values, dimension=3, lag=1)
available_year = series.YEAR.to_numpy()[2:]
assert len(windows)==len(available_year)
assert np.array_equal(windows[0], values[:3])
display(pd.DataFrame({'available_year': available_year[:5],
                      'older': windows[:5,0], 'previous': windows[:5,1], 'current': windows[:5,2]}))

,available_year,older,previous,current
0,1702,5.0,11.0,16.0
1,1703,11.0,16.0,23.0
2,1704,16.0,23.0,36.0
3,1705,23.0,36.0,58.0
4,1706,36.0,58.0,29.0


## Step 3

Record that this is feature construction, not an independently evaluated forecasting model.

Before running: write your expected result and name one way this check could fail.

In [5]:
result = {'scaler_fit_rows': len(train), 'validation_transform_rows': len(valid),
          'time_series_window_count': len(windows), 'first_available_year': int(available_year[0]),
          'forecast_validation_performed': False, 'adjacent_windows_independent': False}

## Keep execution evidence separate from learning evidence

The following saves this reference calculation. Write your own explanation in `my_work/`; no learner score is assigned.

In [6]:
result['stage'] = 11
result['execution_role'] = 'reference_consolidation'
result['data_role'] = 'Real train/validation digit pixels and the bundled historical annual sunspot series; no new forecast claim.'
output = ROOT/'reports/consolidation_v4/11.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Saved reference result:', output.relative_to(ROOT))


Saved reference result: reports/consolidation_v4/11.json
